In [89]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [90]:
df = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']).drop(columns=['timestamp'])  # Load your dataset here
df.head()

,user_id,item_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [91]:
df["user_id"] = df["user_id"]-1
df["item_id"] = df["item_id"]-1

In [92]:
positive_ratings = df[df['rating']>= 4].astype(int)
positive_ratings.head()

,user_id,item_id,rating
5,297,473,4
7,252,464,5
11,285,1013,5
12,199,221,5
16,121,386,5


In [93]:
positive_ratings.nunique()

user_id     942
item_id    1447
rating        2
dtype: int64

In [94]:
positive_ratings.shape

(55375, 3)

In [95]:
positive_ratings["label"] = 1
positive_ratings = positive_ratings.drop(columns=['rating'])
positive_ratings.head()

,user_id,item_id,label
5,297,473,1
7,252,464,1
11,285,1013,1
12,199,221,1
16,121,386,1


In [96]:
negative_ratings = df[df["rating"] <= 3]
negative_ratings['label'] = 0
negative_ratings = negative_ratings.drop(columns=['rating'])
negative_ratings.head()

,user_id,item_id,label
0,195,241,0
1,185,301,0
2,21,376,0
3,243,50,0
4,165,345,0


In [97]:
implicit_df = pd.concat(
    [positive_ratings, negative_ratings],
    ignore_index=True
)

In [98]:
implicit_df = implicit_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [99]:

implicit_df["label"].value_counts()

label
1    55375
0    44625
Name: count, dtype: int64

In [100]:
implicit_df.head()

,user_id,item_id,label
0,12,305,0
1,631,275,0
2,591,846,1
3,616,287,0
4,789,576,0


In [101]:

train_set, test_set = train_test_split(
    implicit_df,
    test_size=0.2,
    random_state=42
)

In [102]:
class MovieLen100k(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user_id = self.data.iloc[idx]['user_id']
        item_id = self.data.iloc[idx]['item_id']
        label = self.data.iloc[idx]['label']
        return torch.tensor(user_id, dtype=torch.long), torch.tensor(item_id, dtype=torch.long), torch.tensor(label, dtype=torch.float)

In [103]:
train_data, test_data = MovieLen100k(train_set), MovieLen100k(test_set)
train_loader, test_loader = DataLoader(train_data, batch_size=256, shuffle=True), DataLoader(test_data, batch_size=256, shuffle=False)

In [104]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_size):
        super(MatrixFactorization, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.item_embedding = nn.Embedding(num_items, embedding_size)

    def forward(self, user_id, item_id):
        user_vector = self.user_embedding(user_id)
        item_vector = self.item_embedding(item_id)
        return (user_vector * item_vector).sum(1)  # Dot product

In [105]:
model = MatrixFactorization(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), embedding_size=32)

In [106]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss = nn.BCEWithLogitsLoss()

In [107]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for user_id, item_id, label in train_loader:

        optimizer.zero_grad()

        logits = model(user_id, item_id) # logit for BCEWithLogitsLoss, not probability

        l = loss(logits, label)

        l.backward()
        optimizer.step()

        total_loss += l.item()

    avg_train_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}")

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for user_id, item_id, label in test_loader:

            logits = model(user_id, item_id)

            l = loss(logits, label)

            total_loss += l.item()

    avg_test_loss = total_loss / len(test_loader)

    print(f"Test Loss: {avg_test_loss:.4f}")

Epoch 1/10, Train Loss: 2.0548
Test Loss: 1.7508
Epoch 2/10, Train Loss: 1.1502
Test Loss: 1.3964
Epoch 3/10, Train Loss: 0.7493
Test Loss: 1.2082
Epoch 4/10, Train Loss: 0.5554
Test Loss: 1.1093
Epoch 5/10, Train Loss: 0.4486
Test Loss: 1.0679
Epoch 6/10, Train Loss: 0.3825
Test Loss: 1.0610
Epoch 7/10, Train Loss: 0.3348
Test Loss: 1.0778
Epoch 8/10, Train Loss: 0.2968
Test Loss: 1.1096
Epoch 9/10, Train Loss: 0.2648
Test Loss: 1.1559
Epoch 10/10, Train Loss: 0.2375
Test Loss: 1.2121


In this notebook, we're doing implicit feedback (0 for <= 3 rating and 1 for >=4 rating) with MF. Since it's a binary classification, we'll be using BCEwithLogitsLoss instead of MSE